# Lab 4 – Logical Reasoning for Planning

**Name:** Mehta Prem Kalpesh 

**ID:** 2024ADPS0851G

## Task 0:

**(a) Initial state**

I = {At(Robot, A), At(Package, A)}

**(b) Goal**

G = {At(Package, C)}

**(c) Available actions**

- **Move(X, Y)** for each connected pair of locations: Move(A, B), Move(B, A), Move(B, C), Move(C, B)
- **PickUp(Package, L)** for L in {A, B, C}
- **Drop(Package, L)** for L in {A, B, C}

**(d) Preconditions and effects**

| Action | Preconditions | Effects |
|---|---|---|
| Move(X, Y) | At(Robot, X) | ¬At(Robot, X), At(Robot, Y) |
| PickUp(Package, L) | At(Robot, L), At(Package, L) | ¬At(Package, L), Holding(Package) |
| Drop(Package, L) | At(Robot, L), Holding(Package) | ¬Holding(Package), At(Package, L) |

(Move is only defined between connected locations: A–B and B–C. There is no direct A–C move.)

### Which actions are initially applicable?

Starting state: I = {At(Robot, A), At(Package, A)}

**1. Is PickUp(Package, A) applicable? Yes.**

Preconditions: At(Robot, A) and At(Package, A).
- At(Robot, A) is in I ✓
- At(Package, A) is in I ✓

All preconditions are satisfied, so I ⊨ Preconditions(PickUp(Package, A)) and the action is applicable.

**2. Is Drop(Package, C) applicable? No.**

Preconditions: At(Robot, C) and Holding(Package).
- At(Robot, C) is not in I ✗ (the robot is at A)
- Holding(Package) is not in I ✗ (the robot has not picked up the package)

Both preconditions fail, so I does not entail Preconditions(Drop(Package, C)) and the action is not applicable.

## Task 1: Construct a Plan by Hand

**Plan:** PickUp(Package, A) → Move(A, B) → Move(B, C) → Drop(Package, C)

| State | Action applied | Facts |
|---|---|---|
| S0 | (initial) | At(Robot, A), At(Package, A) |
| S1 | PickUp(Package, A) | At(Robot, A), Holding(Package) |
| S2 | Move(A, B) | At(Robot, B), Holding(Package) |
| S3 | Move(B, C) | At(Robot, C), Holding(Package) |
| S4 | Drop(Package, C) | At(Robot, C), At(Package, C) |

## Task 2: Ask an LLM to Implement the Planner

## Prompt:

I want to implement a simple planning agent in Python 3 (standard library only, single file).

## Planner specification
Represent a state as a set of logical propositions (e.g. a frozenset of strings such as "At(Robot,A)").
Each action should contain:
- a name;
- positive preconditions;
- negative preconditions;
- positive effects;
- negative effects.

An action is applicable if all of its positive preconditions are in the current state and none of its negative preconditions are in the current state.
When an action is applied:
1. remove its negative effects from the state;
2. add its positive effects to the state.

Use breadth-first search to find a sequence of actions that achieves a specified goal (the goal is satisfied when every goal proposition is in the state).

The program should also:
- detect when no plan exists and print "No plan found";
- print the resulting sequence of actions;
- print the states reached after each action (starting with the initial state).

## Design requirements
- Keep the problem definition (initial state, goal, list of actions) separate from the planner code, so I can change the problem (e.g. remove an action or add a new one) without touching the search.
- The planner should be a function that takes (initial_state, goal, actions) and returns a list of actions, or None if no plan exists.
- Avoid revisiting states already explored.
- Include a main section that runs the planner on the problem below.

## Problem: warehouse robot
A robot must transport a package between three locations: A, B, C.
The only connections are A–B and B–C (in both directions). There is no direct A–C connection.

Initial state: {At(Robot,A), At(Package,A)}
Goal: {At(Package,C)}

Action schemas (ground them over the locations; Move only for connected pairs):

| Action | Preconditions | Effects |
|---|---|---|
| Move(X,Y) | At(Robot,X) | ¬At(Robot,X), At(Robot,Y) |
| PickUp(Package,L) | At(Robot,L), At(Package,L) | ¬At(Package,L), Holding(Package) |
| Drop(Package,L) | At(Robot,L), Holding(Package) | ¬Holding(Package), At(Package,L) |

Explain the implementation and identify any assumptions you make.

In [ ]:
from collections import deque

class Action:
    """Represents a grounded action with preconditions and effects."""
    def __init__(self, name, pos_pre, neg_pre, pos_eff, neg_eff):
        self.name = name
        # Using frozenset for efficient set operations and immutability
        self.pos_pre = frozenset(pos_pre)
        self.neg_pre = frozenset(neg_pre)
        self.pos_eff = frozenset(pos_eff)
        self.neg_eff = frozenset(neg_eff)

    def is_applicable(self, state):
        """
        An action is applicable if all positive preconditions are in the state,
        and no negative preconditions are in the state.
        """
        return self.pos_pre.issubset(state) and self.neg_pre.isdisjoint(state)

    def apply(self, state):
        """
        Applies the action by removing negative effects and adding positive effects.
        """
        return (state - self.neg_eff) | self.pos_eff


def bfs_plan(initial_state, goal, actions):
    """
    Finds the shortest sequence of actions to reach a state that satisfies the goal.
    Returns a list of Action objects, or None if no plan exists.
    """
    # Check if the initial state already satisfies the goal
    if goal.issubset(initial_state):
        return []

    # Queue stores tuples of (current_state, path_of_actions)
    queue = deque([(initial_state, [])])
    
    # Track visited states to avoid cycles. 
    # Works safely because states are frozensets (which are hashable).
    visited = {initial_state}

    while queue:
        current_state, path = queue.popleft()

        for action in actions:
            if action.is_applicable(current_state):
                new_state = action.apply(current_state)
                
                if new_state not in visited:
                    new_path = path + [action]
                    
                    # Check if the new state satisfies our goal
                    if goal.issubset(new_state):
                        return new_path
                    
                    visited.add(new_state)
                    queue.append((new_state, new_path))
                    
    return None  # Queue exhausted without finding a goal


def print_execution(initial_state, plan):
    """Executes the plan and prints the state progression."""
    if plan is None:
        print("No plan found.")
        return

    print("=== Plan Execution ===")
    print(f"Step 0 (Initial state): {sorted(list(initial_state))}")
    
    current_state = initial_state
    for i, action in enumerate(plan, 1):
        current_state = action.apply(current_state)
        print(f"\nAction {i}: {action.name}")
        print(f"Step {i} (State): {sorted(list(current_state))}")


def main():
    # 1. Define Initial State and Goal
    initial_state = frozenset({"At(Robot,A)", "At(Package,A)"})
    goal = frozenset({"At(Package,C)"})

    # 2. Define Grounded Actions
    # We manually ground the schemas (Move, PickUp, Drop) over locations A, B, C.
    actions = [
        # Move Actions (Connections: A-B, B-C in both directions)
        Action("Move(A,B)", pos_pre={"At(Robot,A)"}, neg_pre=set(), 
               pos_eff={"At(Robot,B)"}, neg_eff={"At(Robot,A)"}),
        Action("Move(B,A)", pos_pre={"At(Robot,B)"}, neg_pre=set(), 
               pos_eff={"At(Robot,A)"}, neg_eff={"At(Robot,B)"}),
        Action("Move(B,C)", pos_pre={"At(Robot,B)"}, neg_pre=set(), 
               pos_eff={"At(Robot,C)"}, neg_eff={"At(Robot,B)"}),
        Action("Move(C,B)", pos_pre={"At(Robot,C)"}, neg_pre=set(), 
               pos_eff={"At(Robot,B)"}, neg_eff={"At(Robot,C)"}),
        
        # PickUp Actions
        Action("PickUp(Package,A)", pos_pre={"At(Robot,A)", "At(Package,A)"}, neg_pre=set(), 
               pos_eff={"Holding(Package)"}, neg_eff={"At(Package,A)"}),
        Action("PickUp(Package,B)", pos_pre={"At(Robot,B)", "At(Package,B)"}, neg_pre=set(), 
               pos_eff={"Holding(Package)"}, neg_eff={"At(Package,B)"}),
        Action("PickUp(Package,C)", pos_pre={"At(Robot,C)", "At(Package,C)"}, neg_pre=set(), 
               pos_eff={"Holding(Package)"}, neg_eff={"At(Package,C)"}),
        
        # Drop Actions
        Action("Drop(Package,A)", pos_pre={"At(Robot,A)", "Holding(Package)"}, neg_pre=set(), 
               pos_eff={"At(Package,A)"}, neg_eff={"Holding(Package)"}),
        Action("Drop(Package,B)", pos_pre={"At(Robot,B)", "Holding(Package)"}, neg_pre=set(), 
               pos_eff={"At(Package,B)"}, neg_eff={"Holding(Package)"}),
        Action("Drop(Package,C)", pos_pre={"At(Robot,C)", "Holding(Package)"}, neg_pre=set(), 
               pos_eff={"At(Package,C)"}, neg_eff={"Holding(Package)"})
    ]

    # 3. Run the Planner
    plan = bfs_plan(initial_state, goal, actions)

    # 4. Output the results
    print_execution(initial_state, plan)


if __name__ == "__main__":
    main()

No plan found.


## Task 2: Think About It

| Idea from the specification | Where it appears in the program | What it does |
|---|---|---|
| **Preconditions** → When is an action applicable? | `Action.is_applicable(state)` (line 18), called from `bfs_plan` (line 47) | Returns `True` only if every positive precondition is in the state (`pos_pre.issubset(state)`) and no negative precondition is in the state (`neg_pre.isdisjoint(state)`). This is the check S ⊨ Preconditions(a). |
| **Effects** → How does the state change? | `Action.apply(state)` (line 24) | Returns `(state - neg_eff) \| pos_eff`: the negative effects are removed first, then the positive effects are added, which is the order the specification gave. The original state is not modified; a new state is returned. |
| **Goal** → When does planning terminate? | The goal test `goal.issubset(new_state)` (line 54), plus `goal.issubset(initial_state)` (line 33) | The goal itself is the data `goal = frozenset({"At(Package,C)"})` in `main()`. Planning terminates when a newly generated state contains every goal proposition, and the path that led to it is returned. If the initial state already satisfies the goal, an empty plan is returned. If the queue empties without a goal state, the function returns `None`, which is reported as "No plan found". |
| **BFS** → How are alternative plans explored? | `bfs_plan` (lines 27–60): a FIFO `deque` with `popleft()` (line 44), a `for` loop over all actions (line 46), and the `visited` set (line 41) | Each applicable action produces a successor state that is added to the back of the queue together with the action sequence that reached it. Because the queue is first-in, first-out, all plans of length k are explored before any plan of length k+1, so the first plan found is a shortest one. The `visited` set stops the search from revisiting states, for example in the loop A → B → A → B. |

**Assumptions visible in the generated code**
- A state is a `frozenset` of proposition strings, and a proposition that is absent from the state is treated as false (closed-world assumption).
- The actions were grounded by hand over the locations A, B and C, with `Move` only on the connected pairs A–B and B–C.
- Every action has the same cost, so BFS finds a plan with the fewest actions.

## Task 3: Test the Generated Planner

### Test A: Solvable problem
- **Initial state:** {At(Robot,A), At(Package,A)}
- **Goal:** {At(Package,C)}
- **Plan found?** Yes
- **Plan:** PickUp(Package,A) → Move(A,B) → Move(B,C) → Drop(Package,C)
- **Valid?** Yes. The validator confirmed that every action's preconditions held in the state before it. The final state is {At(Robot,C), At(Package,C)}, which satisfies the goal.
- The plan has 4 actions, which is the minimum possible (one pick-up, two moves, one drop). It is identical to my hand-built plan from Task 1, as expected from BFS.

### Test B: Impossible problem
- **Change:** all PickUp actions removed.
- **Initial state:** {At(Robot,A), At(Package,A)}
- **Goal:** {At(Package,C)}
- **Plan found?** No. `bfs_plan` returned `None`, which the program reports as "No plan found".
- **Valid?** Not applicable, since no plan was invented. Without PickUp, the package can never leave A.

### Test C: Irrelevant actions
- **C1 (control):** only Move actions, goal {At(Robot,C)}. Plan found: Move(A,B) → Move(B,C). Valid. This shows the robot can reach C.
- **C2:** same Move-only actions, goal {At(Package,C)}. No plan found. The robot reaches C in C1, but the planner does not treat that as the package being at C.
- **C3:** the full problem plus an extra robot-only action, Wander(A), which does not move the package. The plan is the same 4 actions as in Test A, Wander is not used, and the plan is valid.

### Summary
| Test | Plan found | Plan valid |
|---|---|---|
| A: solvable | Yes (4 actions) | Yes |
| B: PickUp removed | No | n/a (correct) |
| C1: robot to C | Yes (2 actions) | Yes |
| C2: package to C, Move-only | No | n/a (correct) |
| C3: extra Wander action | Yes (4 actions) | Yes |
| Extra: goal already true | Yes (empty plan) | Yes |
| Extra: C unreachable | No | n/a (correct) |

No bugs were found in the generated planner.

## Task 4: Logic and Search

**Completed pipeline**

Current state

↓

Check action preconditions (S ⊨ Preconditions(a)?)

↓

**Select the applicable actions**

↓

Generate successor state S′ = Apply(S, a)

↓

Search over alternatives (add successors to the BFS queue, skip visited states)

↓

Goal? (if goal ⊆ S′, return the plan; otherwise continue with the next state in the queue)


**Where logical reasoning is used**
`Action.is_applicable(state)` decides whether the preconditions are entailed by the current state: all positive preconditions are in the state and no negative precondition is. `Action.apply(state)` then computes how the state changes: negative effects are removed and positive effects are added. Together they define what is possible and what its consequences are.

**Where search is used**
`bfs_plan` decides which sequences of applicable actions to try. It keeps a FIFO queue of (state, plan so far) pairs, expands states in order of plan length, and uses a `visited` set to avoid repeating states. The goal test `goal ⊆ new_state` stops the search. Because BFS tries all plans of length k before length k+1, the first plan it finds uses the fewest actions.

**How they work together**
Search alone would have no way to know which actions are allowed or what they do, so it would try meaningless moves. Logic alone can say whether one action is legal but cannot choose among the many legal sequences. In the planner, logic filters the actions at each state down to the legal ones and produces the resulting states, and search organises those states into a systematic exploration until one satisfies the goal.

Logic determines what is possible; search determines what to try.

## Task 5 (Optional): Can the LLM Verify Its Own Plan?

**Method.** I asked an LLM to explain the plan in two prompts, then compared each explanation with the state transitions.

### Prompt 1: the lab's prompt on the real plan
Plan: PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C). Schema provided in the prompt.

The LLM listed the preconditions and effects of every action and the state after each step. All four states (S1–S4), all preconditions and the goal check matched exactly.

**Result:** the explanation was accurate.

### Prompt 2: challenge with an invalid plan
Fresh chat, no schema, and a deliberately misleading prompt ("explain why this plan is valid") for the plan Move(A,B), PickUp(Package,B), Move(B,C), Drop(Package,C).

The LLM did not accept the premise. It found that PickUp(Package,B) fails because the package is still at A, said the plan is invalid, and suggested the correct plan PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C).

**Observation:** the LLM added a precondition that was not specified ("robot must not already be holding a package"). It did not change the verdict, but it shows that an explanation can introduce assumptions that were never given.

### Question: which should you trust more?
(a) the LLM's explanation; (b) the independently executed state transitions?

**Which to trust more: (b) the independently executed state transitions.**

The executed transitions apply the stated preconditions and effects mechanically: they are deterministic, reproducible, and cannot be influenced by how a question is phrased. An LLM explanation is generated text. It can be fluent and convincing whether or not it is correct, and nothing in the explanation itself reveals an error. In Prompt 1 the explanation was correct, but I could only know that by comparing it with the executed states. In Prompt 2 the model could have rationalised an invalid plan because the prompt asked it to "explain why it is valid"; it happened to catch the error, but that is not guaranteed.

The independent check should itself be tested, which is why I ran the validator on invalid plans (the lab's PickUp(Package,B) example and a Move(A,C)) and on normal and edge cases before relying on it. A generated explanation is not the same as an independent verification.

## Task 6: Plan with a Key and a Locked Door

### (a) Shortest valid plan by hand

**Plan:** PickKey → Move(A,B) → Unlock → Move(B,C)

| State | Action applied | Facts |
|---|---|---|
| S0 | (initial) | At(A), KeyAt(A) |
| S1 | PickKey | At(A), HasKey |
| S2 | Move(A,B) | At(B), HasKey |
| S3 | Unlock | At(B), HasKey, Unlocked |
| S4 | Move(B,C) | At(C), HasKey, Unlocked |

### (b) BFS planner on the locked-door domain

The `Action` class and `bfs_plan` from the warehouse task were reused without any change. Only the initial state, the goal and the six-action table were replaced.

- Initial state: {At(A), KeyAt(A)}
- Goal: {At(C)}

**Plan found by BFS:** PickKey → Move(A,B) → Unlock → Move(B,C) (4 actions)

**State trace**

| Step | Action | State |
|---|---|---|
| 0 | (initial) | At(A), KeyAt(A) |
| 1 | PickKey | At(A), HasKey |
| 2 | Move(A,B) | At(B), HasKey |
| 3 | Unlock | At(B), HasKey, Unlocked |
| 4 | Move(B,C) | At(C), HasKey, Unlocked |

The BFS plan is identical to the hand-built plan, and its length (4) matches the minimum argued in (a).

In [6]:
"""Locked-door domain (Task 6). Reuses Action and bfs_plan from planner.py unchanged."""

I = frozenset({"At(A)", "KeyAt(A)"})
G = frozenset({"At(C)"})


def A(name, pre, add, delete):
    return Action(name, pre, set(), add, delete)


def door_actions():
    return [
        A("Move(A,B)", {"At(A)"}, {"At(B)"}, {"At(A)"}),
        A("Move(B,A)", {"At(B)"}, {"At(A)"}, {"At(B)"}),
        A("PickKey", {"At(A)", "KeyAt(A)"}, {"HasKey"}, {"KeyAt(A)"}),
        A("Unlock", {"At(B)", "HasKey"}, {"Unlocked"}, set()),
        A("Move(B,C)", {"At(B)", "Unlocked"}, {"At(C)"}, {"At(B)"}),
        A("Move(C,B)", {"At(C)", "Unlocked"}, {"At(B)"}, {"At(C)"}),
    ]


if __name__ == "__main__":
    print("--- Task 6(b): BFS on the locked-door domain ---")
    plan = bfs_plan(I, G, door_actions())
    print("plan:", [a.name for a in plan], "| length", len(plan))
    print_execution(I, plan)

--- Task 6(b): BFS on the locked-door domain ---
plan: ['PickKey', 'Move(A,B)', 'Unlock', 'Move(B,C)'] | length 4
=== Plan Execution ===
Step 0 (Initial state): ['At(A)', 'KeyAt(A)']

Action 1: PickKey
Step 1 (State): ['At(A)', 'HasKey']

Action 2: Move(A,B)
Step 2 (State): ['At(B)', 'HasKey']

Action 3: Unlock
Step 3 (State): ['At(B)', 'HasKey', 'Unlocked']

Action 4: Move(B,C)
Step 4 (State): ['At(C)', 'HasKey', 'Unlocked']


### (c) PickKey removed

**Result:** no plan exists. BFS exhausts the search and returns "No plan found".

**Explanation:** the dependencies form a chain in which each fact has exactly one source.
- HasKey is added only by PickKey, which was removed, so HasKey is never true.
- Unlock requires HasKey, so it is never applicable and Unlocked is never true.
- Move(B,C) requires Unlocked, so it is never applicable and At(C) is never true.

Only two states are reachable: {At(A), KeyAt(A)} and {At(B), KeyAt(A)}. The robot can walk between A and B but can never get past the locked door.

### (d) Validating Move(A,B), Unlock, Move(B,C)

| Step | Action | Preconditions | State before | Result |
|---|---|---|---|---|
| 1 | Move(A,B) | At(A) | {At(A), KeyAt(A)} | ✓ applicable → {At(B), KeyAt(A)} |
| 2 | Unlock | At(B), HasKey | {At(B), KeyAt(A)} | ✗ not applicable |

**First invalid action:** Unlock (step 2).
**Unsatisfied precondition:** HasKey. At(B) is satisfied. The robot moved to B without picking up the key first, so the plan is invalid, and Move(B,C) is never reached.

In [8]:
"""Task 6(c), 6(d) and the independent validator for the locked-door domain."""

# Independent encoding of the lab's action table: name -> (preconditions, add, delete).
# Deliberately NOT built from Action objects, so it can check the planner's encoding.
TABLE = {
    "Move(A,B)": ({"At(A)"}, {"At(B)"}, {"At(A)"}),
    "Move(B,A)": ({"At(B)"}, {"At(A)"}, {"At(B)"}),
    "PickKey":   ({"At(A)", "KeyAt(A)"}, {"HasKey"}, {"KeyAt(A)"}),
    "Unlock":    ({"At(B)", "HasKey"}, {"Unlocked"}, set()),
    "Move(B,C)": ({"At(B)", "Unlocked"}, {"At(C)"}, {"At(B)"}),
    "Move(C,B)": ({"At(C)", "Unlocked"}, {"At(B)"}, {"At(C)"}),
}


def validate_door_plan(initial, goal, names):
    """Replay `names` from `initial`. Returns (ok, log, first_error)."""
    state, log = set(initial), []
    for i, name in enumerate(names, 1):
        if name not in TABLE:
            return False, log, f"step {i}: {name} is not one of the six permitted actions"
        pre, add, dele = TABLE[name]
        missing = pre - state
        if missing:
            return False, log, (f"step {i}: {name} is not applicable in {sorted(state)}; "
                                f"unsatisfied precondition(s): {sorted(missing)}")
        state = (state - dele) | add
        assert sum(p.startswith("At(") for p in state) == 1, "invariant: exactly one At(.)"
        log.append(f"step {i}: {name} ok -> {sorted(state)}")
    if not set(goal) <= state:
        return False, log, f"plan runs but goal {sorted(goal)} not reached; final state {sorted(state)}"
    return True, log, None


if __name__ == "__main__":
    print("--- Task 6(c): PickKey removed ---")
    no_key = [a for a in door_actions() if a.name != "PickKey"]
    plan = bfs_plan(I, G, no_key)
    print("planner result:", "No plan found" if plan is None else [a.name for a in plan])
    # What is reachable at all? (explains why)
    reach, frontier = {I}, [I]
    while frontier:
        s = frontier.pop()
        for a in no_key:
            if a.is_applicable(s):
                t = a.apply(s)
                if t not in reach:
                    reach.add(t); frontier.append(t)
    print("reachable states:", [sorted(s) for s in reach])
    print("HasKey ever true:", any("HasKey" in s for s in reach),
          "| Unlocked ever true:", any("Unlocked" in s for s in reach),
          "| At(C) ever true:", any("At(C)" in s for s in reach))

    print("\n--- Task 6(d): validate Move(A,B), Unlock, Move(B,C) ---")
    ok, log, err = validate_door_plan(I, G, ["Move(A,B)", "Unlock", "Move(B,C)"])
    print("\n".join(log)); print("valid:", ok); print("first error:", err)

    print("\n--- Validator sanity checks ---")
    ok, log, err = validate_door_plan(I, G, ["PickKey", "Move(A,B)", "Unlock", "Move(B,C)"])
    print("BFS plan valid:", ok)
    assert ok
    ok, _, err = validate_door_plan(I, G, ["Move(A,C)"])
    print("Move(A,C):", err); assert not ok
    ok, _, err = validate_door_plan(I, G, ["PickKey", "Move(A,B)", "Unlock"])
    print("plan stopping at Unlock:", err); assert not ok

--- Task 6(c): PickKey removed ---
planner result: No plan found
reachable states: [['At(B)', 'KeyAt(A)'], ['At(A)', 'KeyAt(A)']]
HasKey ever true: False | Unlocked ever true: False | At(C) ever true: False

--- Task 6(d): validate Move(A,B), Unlock, Move(B,C) ---
step 1: Move(A,B) ok -> ['At(B)', 'KeyAt(A)']
valid: False
first error: step 2: Unlock is not applicable in ['At(B)', 'KeyAt(A)']; unsatisfied precondition(s): ['HasKey']

--- Validator sanity checks ---
BFS plan valid: True
Move(A,C): step 1: Move(A,C) is not one of the six permitted actions
plan stopping at Unlock: plan runs but goal ['At(C)'] not reached; final state ['At(B)', 'HasKey', 'Unlocked']


## Task 7: Compare LLM Plans Across Two Domains

**Method.** I gave an LLM the exact initial state {At(A), KeyAt(A)}, the goal {At(C)}, and the six-action table, without showing the BFS solution, and asked for only a sequence of the permitted actions. [Name the LLM you used.] I checked the proposed plan with `validate_door_plan`, a validator written from the lab's action table that does not use the planner's `Action` or `bfs_plan`.

**LLM's plan:** PickKey, Move(A,B), Unlock, Move(B,C)

| Check | Result |
|---|---|
| Every action uses only the six permitted actions | Yes |
| Every action's preconditions satisfied in the state where it runs | Yes (steps 1–4) |
| Goal At(C) reached | Yes, final state {At(C), HasKey, Unlocked} |
| Same length as the BFS shortest plan | Yes (4 = 4); the plans are identical |

**Revision:** not needed, since the first plan was valid, so no error was fed back to the LLM.

### Think About It

**Does a plausible narrative guarantee that an action's preconditions are satisfied?**
No. Move(A,B), Unlock, Move(B,C) reads like a sensible story: go to B, unlock the door, go through. But when the plan is replayed, Unlock runs in the state {At(B), KeyAt(A)}, and its precondition HasKey is not satisfied because the key was never picked up. A narrative describes what should happen; only the actual state at that step shows whether the preconditions hold.

**What can an independent state-transition checker establish that an LLM explanation cannot?**
The checker applies the stated rules to the actual state, step by step, and deterministically establishes whether each action's preconditions hold, which action fails first and why, and whether the goal state is reached. An LLM explanation is generated text, so it can be fluent and wrong, and it cannot confirm itself. The checker is only as reliable as the rules it encodes, so it should also be tested on known-invalid plans, as I did.

## Reflection Questions

1. Specifying preconditions and effects first removes ambiguity, so the LLM is not left to guess what the actions do. It also gives me an independent reference (my hand plan) to check the generated code against; I cannot verify what I have not specified.

2. If the planner skipped the precondition check, Move(B,C) would be "applicable" while the robot is at A. Applying it would add At(Robot,C) without the robot ever being at B, leaving the state with the robot in two places. Similarly, skipping the Holding(Package) check would let Drop(Package,C) alone "solve" the problem.

3. A plan can look sensible and still be invalid if some action's preconditions are not satisfied in the state where it runs. For example, Move(A,B), PickUp(Package,B) looks natural, but the package is still at A. Validity is a property of the state sequence, not of how reasonable the plan seems.

4. The LLM contributed the planner implementation (Action class, BFS and the grounded actions).

5. I checked the 10 grounded actions against the schema (including that there is no Move(A,C)), compared the BFS plan with my hand plan, ran tests A, B and C, replayed every plan with an independent validator, and compared the LLM's explanations with the real state transitions.

6. Logical reasoning is used to decide whether an action is applicable (S ⊨ Preconditions(a)), to compute the successor state from the action's effects, and to test whether a state satisfies the goal.

7. Planning is search in a state space. The states are sets of propositions, the initial state is the start node, and the goal test is the condition `goal ⊆ state`. The actions applicable in a state generate the successors (edges), and the resulting plan is the path from start to goal. The BFS used here is the same uninformed search studied in the previous module, with unit cost per action, so it returns a shortest plan. Other search algorithms, such as DFS, uniform-cost search or A* with a heuristic, could be used instead. The difference is that the graph isn't given in advance: logical reasoning (preconditions and effects) computes the successors of each state as the search goes.